# Projekt: GPT-2 laden & Architektur inspizieren
## AI Frameworks & Tools -- Block 1 -- Projektanwendung

**Ziel**: Das Modell kennenlernen, das ihr das Semester ueber untersucht.  
**Tools**: PyTorch, HuggingFace Transformers, Pandas, Matplotlib

---

## 1. Modell laden & Architektur verstehen

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

# GPT-2 laden (ca. 500 MB, wird beim ersten Mal heruntergeladen)
print("Lade GPT-2... (beim ersten Mal dauert der Download ca. 1-2 Min)")
model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model.eval()

print(model)

print("GPT-2 geladen")

print("=== GPT-2 Architektur ===")
print(f"  Transformer-Bloecke (Layers): {model.config.n_layer}")
print(f"  Hidden Dimension:             {model.config.n_embd}")
print(f"  Attention Heads pro Layer:     {model.config.n_head}")
print(f"  Head Dimension (berechnet):    {model.config.n_embd // model.config.n_head}")
print(f"  Vokabulargroesse:              {model.config.vocab_size}")
print(f"  Max. Sequenzlaenge:            {model.config.n_positions}")

**AUFGABE**: Fuelle die Tabelle aus:

| Parameter | Wert |
|-----------|------|
| Anzahl Layers | |
| Hidden Dimension | |
| Attention Heads | |
| Head Dimension | |
| Vokabulargroesse | |
| Max. Sequenzlaenge | |

### 1.2 Parameteranalyse mit Pandas

In [ ]:
import pandas as pd

param_data = []
for name, param in model.named_parameters():
    parts = name.split(".")
    if len(parts) > 1 and parts[0] == "transformer":
        comp_type = parts[1]
    else:
        comp_type = parts[0]

    param_data.append({
        "name": name,
        "shape": str(list(param.shape)),
        "params": param.numel(),
        "component": comp_type,
    })

df_params = pd.DataFrame(param_data)

total_params = df_params["params"].sum()
print(f"Gesamt: {total_params:,} Parameter ({total_params/1e6:.1f}M)")
print(f"\nParameter pro Komponente:")

comp_summary = (
    df_params.groupby("component")["params"]
    .sum()
    .sort_values(ascending=False)
)
for comp, n in comp_summary.items():
    print(f"  {comp:10s}: {n:>12,} ({n/total_params*100:.1f}%)")

In [ ]:
# Detailansicht: Alle Parameter in Layer 0
print("Parameter in Transformer Block 0:\n")
layer0 = df_params[df_params["name"].str.contains("transformer.h.0")]
for _, row in layer0.iterrows():
    print(f"  {row['name']:50s} {row['shape']:20s} {row['params']:>10,}")

**AUFGABE**: Beantworte:

1. Wie viele Parameter hat GPT-2 insgesamt? ...
2. Welche Komponente hat die meisten Parameter? ...
3. Wie viel Prozent entfaellt auf das Embedding (wte + wpe)? ...
4. Wie viel auf die MLP-Schichten? ...

---
## 2. Embedding-Raum erkunden

In [ ]:
# Token-Embedding-Matrix
embeddings = model.transformer.wte.weight.detach()
print(f"Embedding Matrix: {embeddings.shape}")
print(f"  = {embeddings.shape[0]:,} Tokens x {embeddings.shape[1]} Dimensionen")

In [ ]:
def find_similar(word, top_k=5):
    """Finde aehnlichste Tokens im Embedding-Raum via Cosine Similarity."""
    token_id = tokenizer.encode(word)[0]
    word_emb = embeddings[token_id]

    sims = torch.nn.functional.cosine_similarity(
        word_emb.unsqueeze(0), embeddings, dim=1
    )

    top = torch.topk(sims, k=top_k + 1)

    print(f"Aehnlichste Tokens zu '{word.strip()}' (ID {token_id}):")
    for idx, sim in zip(top.indices, top.values):
        if idx != token_id:
            print(f"  '{tokenizer.decode(idx)}': {sim.item():.4f}")

In [ ]:
find_similar(" Paris")
print()
find_similar(" king")
print()
find_similar(" Python")
print()
find_similar(" happy")

In [ ]:
# AUFGABE: Teste mit einem eigenen Wort
# find_similar(" DEIN_WORT")

**AUFGABE**: Dokumentiere:

| Eingabe-Token | Top-3 aehnlichste | Ueberraschend? |
|---------------|-------------------|----------------|
| Paris | | |
| king | | |
| Python | | |
| happy | | |
| [eigenes] | | |

### 2.2 Bonus: Analogie-Test

In [ ]:
def analogy(a, b, c, top_k=5):
    """a ist zu b wie c ist zu ???  (b - a + c)"""
    id_a = tokenizer.encode(a)[0]
    id_b = tokenizer.encode(b)[0]
    id_c = tokenizer.encode(c)[0]

    result_vec = embeddings[id_b] - embeddings[id_a] + embeddings[id_c]

    sims = torch.nn.functional.cosine_similarity(
        result_vec.unsqueeze(0), embeddings, dim=1
    )

    for idx in [id_a, id_b, id_c]:
        sims[idx] = -1

    top = torch.topk(sims, k=top_k)
    print(f"'{a.strip()}' : '{b.strip()}' = '{c.strip()}' : ???")
    for idx, sim in zip(top.indices, top.values):
        print(f"  '{tokenizer.decode(idx)}': {sim.item():.4f}")

In [ ]:
# king - man + woman = ???
analogy(" man", " king", " woman")
print()

# Paris - France + Germany = ???
analogy(" France", " Paris", " Germany")

In [ ]:
# AUFGABE: Finde 2 Analogien die funktionieren und 2 die nicht
# analogy(" ...", " ...", " ...")

---
## 3. Euer Phaenomen testen

### 3.1 Clean- und Corrupted-Prompts definieren

**AUFGABE**: Definiert mindestens 5 Prompt-Paare fuer euer Phaenomen.  
Wichtig: Minimaler Unterschied -- nur das Phaenomen variiert\!

In [ ]:
# AUFGABE: Definiert eure Prompt-Paare
prompt_pairs = [
    # ("Clean-Prompt", "Corrupted-Prompt"),
    # Beispiel Faktenwissen:
    # ("The capital of France is", "The capital of Xyzland is"),
    # Beispiel Negation:
    # ("The movie was really good", "The movie was not really good"),
]

### 3.2 Modellvorhersagen

In [ ]:
def get_top_predictions(prompt, top_k=5):
    """Top-k naechste Token-Vorhersagen fuer einen Prompt."""
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1]

    probs = torch.softmax(logits, dim=-1)
    top = torch.topk(probs, k=top_k)

    results = []
    for prob, idx in zip(top.values, top.indices):
        token = tokenizer.decode(idx)
        results.append((token, prob.item()))
    return results

In [ ]:
# Vorhersagen fuer alle Paare
if prompt_pairs:
    print("=" * 70)
    for i, (clean, corrupted) in enumerate(prompt_pairs, 1):
        print(f"\nPaar {i}:")

        preds_clean = get_top_predictions(clean)
        preds_corr = get_top_predictions(corrupted)

        print(f"  CLEAN:     '{clean}'")
        print(f"  Top-5:     {[(t, f'{p:.3f}') for t, p in preds_clean]}")
        print(f"  CORRUPTED: '{corrupted}'")
        print(f"  Top-5:     {[(t, f'{p:.3f}') for t, p in preds_corr]}")
        print("-" * 70)
else:
    print("Bitte definiere zuerst prompt_pairs in der Zelle oben")
    print("Beispiel:")
    print('  prompt_pairs = [("The capital of France is", "The capital of Xyzland is")]')

### 3.3 Erste Beobachtungen

**AUFGABE**: Notiert in 3-5 Saetzen:

1. Zeigt das Modell das erwartete Verhalten bei den Clean-Prompts? ...
2. Wie unterscheiden sich die Vorhersagen bei Clean vs. Corrupted? ...
3. Erste Intuition -- was koennte im Modell passieren? ...

---
## Naechste Schritte

- **Hausaufgabe**: Phaenomen-Steckbrief schreiben (1 Seite Markdown)
- **Block 2 (17.04.)**: Scikit-Learn + MLflow -> Linear Probes trainieren
- **Frage fuer naechstes Mal**: In welcher Schicht steckt euer Phaenomen?